<a href="https://colab.research.google.com/github/SamanTarique/flyrank-01-ml-2026/blob/main/work%20/notebook/copy_w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb Huggingface_hub


In [ ]:
import os ,getpass

HF_token=os.environ.get('saman_tech')
if not HF_token:
    try:
        from google.colab import userdata
        HF_token = userdata.get('saman_tech')
    except Exception:
        pass
HF_token = HF_token or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [ ]:
import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

files_list_df = con.sql(f"""
    SELECT file
    FROM glob('{REL}/*.parquet')
""").df()

print("--- Data Warehouse Files ---")
for file_name in files_list_df['file']:
    print(file_name)


--- Data Warehouse Files ---
hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Answer:** Unit of Analysis: One row represents a single piece of published content (a specific URL) and its search performance metrics (like clicks, impressions, and position).
Time Window: A single mid-panel calendar month (e.g., March 2026). We are strictly using a mid-panel month to develop our logic and avoid data leakage from the final sealed test month.

In [ ]:
relevant_files_for_verification = files_list_df[files_list_df['file'].isin([
    f"{REL}/dim_content.parquet",
    f"{REL}/fact_content_query_90d.parquet"
])]['file'].tolist()
tabel=relevant_files_for_verification

for table_path in tabel:
    print(f"\n--- Data for {table_path.split('/')[-1]} ---")

    df = con.sql(f"SELECT * FROM '{table_path}' LIMIT 5").df()
    display(df)



--- Data for dim_content.parquet ---


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False



--- Data for fact_content_query_90d.parquet ---


,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,...,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,...,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Answer:**

**Label :**
- avg_position_last30
- avg_position_prev30
- Derived: `declined = avg_position_last30 > avg_position_prev30`
- Why: This is the outcome we're ranking content by — did it get worse. It's a proxy, not a true label, since we don't have real refresh-outcome history.

**Feature:**
- last_optimized_date (→ days since refresh)
- char_count
- word_count
- content_visible_query_count
- rare_query_count
- content_total_impressions_90d
- avg_position_90d
- Why: Known before the last30/prev30 split — safe to use as predictors, no leakage.

**Context:**
- content_type
- is_published
- window_start
- window_end
- optimization_eligible_date
- Why: Used for filtering/segmenting (the IS TRUE availability check), not fed to the model.

**Excluded:**
- client_hash_id
- content_hash_id
- query_hash_id
- url_hash_id
- provider_used
- model_used
- is_deleted
- Why: Pure identifiers or pipeline metadata — no predictive signal, or filtered out before scoring.

In [ ]:
buckets = {
    "label_proxy": ["avg_position_last30", "avg_position_prev30"],
    "feature": ["last_optimized_date", "char_count", "word_count",
                "content_visible_query_count", "rare_query_count",
                "content_total_impressions_90d", "avg_position_90d"],
    "context": ["content_type", "is_published", "window_start",
                "window_end", "optimization_eligible_date"],
    "excluded": ["client_hash_id", "content_hash_id", "query_hash_id",
                 "url_hash_id", "provider_used", "model_used", "is_deleted"],
}

for name, cols in buckets.items():
    print(f"{name.upper()} ({len(cols)}): {cols}")

LABEL_PROXY (2): ['avg_position_last30', 'avg_position_prev30']
FEATURE (7): ['last_optimized_date', 'char_count', 'word_count', 'content_visible_query_count', 'rare_query_count', 'content_total_impressions_90d', 'avg_position_90d']
CONTEXT (5): ['content_type', 'is_published', 'window_start', 'window_end', 'optimization_eligible_date']
EXCLUDED (7): ['client_hash_id', 'content_hash_id', 'query_hash_id', 'url_hash_id', 'provider_used', 'model_used', 'is_deleted']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

#### **GRAIN**(empty result means true, rows mean duplicates exist.

In [ ]:
con.sql(f"""
    SELECT content_hash_id, COUNT(*) AS row_count
    FROM '{REL}/dim_content.parquet'
    GROUP BY content_hash_id
    HAVING COUNT(*) > 1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,row_count


### **WINDOW+COUNT**

In [ ]:
con.sql(f"""
    SELECT MIN(window_start) AS min_start,
           MAX(window_start) AS max_start,
           MIN(window_end) AS min_end,
           MAX(window_end) AS max_end
    FROM '{REL}/fact_content_query_90d.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_start,max_start,min_end,max_end
0,2026-04-02,2026-04-02,2026-06-30,2026-06-30


#### Verifying `dim_content` Content Availability (`is_published`, `is_deleted`)

In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE is_published IS TRUE AND is_deleted IS NOT TRUE) AS available_rows
    FROM '{REL}/dim_content.parquet'
""").df()

,total_rows,available_rows
0,519606,411540


#### **Missing Values**

In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(char_count) AS missing_char_count,
        COUNT(*) - COUNT(last_optimized_date) AS missing_last_optimized_date,
        COUNT(*) - COUNT(content_type) AS missing_content_type
    FROM '{REL}/dim_content.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_char_count,missing_last_optimized_date,missing_content_type
0,519606,177768,474210,0


#### Verifying `dim_content` Missing Values (`char_count`, `last_optimized_date`, `content_type`)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
print("""
What this data can NEVER tell us:

True Content Value Beyond Google (GSC-Only): This dataset is built purely on Google Search Console metrics. It tells us nothing about traffic from social media, direct visits, or actual on-page user engagement (like time spent reading or bounce rates).

Perfect Apples-to-Apples Comparisons (Unbalanced History): We are scoring URLs with entirely different lifespans. A page published 2 years ago has a massive historical advantage over a page published 2 months ago, making strict historical comparisons unbalanced.

Independent Events (Window Overlaps): Because we use rolling 30-day and 90-day performance metrics, consecutive months heavily overlap. The data points aren't fully independent, which means an anomaly in one week will echo across multiple evaluation windows.""")


What this data can NEVER tell us:

True Content Value Beyond Google (GSC-Only): This dataset is built purely on Google Search Console metrics. It tells us nothing about traffic from social media, direct visits, or actual on-page user engagement (like time spent reading or bounce rates).

Perfect Apples-to-Apples Comparisons (Unbalanced History): We are scoring URLs with entirely different lifespans. A page published 2 years ago has a massive historical advantage over a page published 2 months ago, making strict historical comparisons unbalanced.

Independent Events (Window Overlaps): Because we use rolling 30-day and 90-day performance metrics, consecutive months heavily overlap. The data points aren't fully independent, which means an anomaly in one week will echo across multiple evaluation windows.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.